In [8]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
data = {
    "question1": [
        "How can I learn Python?",
        "What is the capital of India?",
        "How do I make tea?",
        "What is machine learning?",
        "How can I improve my English?",
        "What is photosynthesis?",
        "How can I lose weight?",
        "What is data science?",
        "How do I apply for a passport?",
        "What causes rain?",
        "How can I learn Python?",
        "What is the capital of India?",
        "How do I make tea?",
        "What is machine learning?",
        "How can I improve my English?",
        "What is photosynthesis?",
        "How can I lose weight?",
        "What is data science?",
        "How do I apply for a passport?",
        "What causes rain?"
    ],

    "question2": [
        "What is the best way to learn Python?",
        "Which city is the capital of India?",
        "How can I prepare tea?",
        "Explain machine learning.",
        "What are ways to improve my English?",
        "Explain the process of photosynthesis.",
        "What are some ways to reduce weight?",
        "Can you explain data science?",
        "What is the process to get a passport?",
        "Why does rainfall happen?",
        "What is Python used for?",
        "What is the population of India?",
        "How do I bake a cake?",
        "What is deep learning?",
        "How can I learn French?",
        "What is cellular respiration?",
        "How can I gain muscle?",
        "What does a data analyst do?",
        "How do I get a driving licence?",
        "What causes earthquakes?"
    ],

    "is_duplicate": [
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0
    ]
}

df = pd.DataFrame(data)

## Explore the Dataset

In [10]:
print("First five rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate and non-duplicate counts:")
print(df["is_duplicate"].value_counts())

First five rows:
                       question1                              question2  \
0        How can I learn Python?  What is the best way to learn Python?   
1  What is the capital of India?    Which city is the capital of India?   
2             How do I make tea?                 How can I prepare tea?   
3      What is machine learning?              Explain machine learning.   
4  How can I improve my English?   What are ways to improve my English?   

   is_duplicate  
0             1  
1             1  
2             1  
3             1  
4             1  

Dataset shape:
(20, 3)

Missing values:
question1       0
question2       0
is_duplicate    0
dtype: int64

Duplicate and non-duplicate counts:
is_duplicate
1    10
0    10
Name: count, dtype: int64


## Text Preprocessing

In [13]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["q1_clean"] = df["question1"].apply(clean_text)
df["q2_clean"] = df["question2"].apply(clean_text)

print("Cleaned questions:")
print(df[["q1_clean", "q2_clean"]].head())

Cleaned questions:
                       q1_clean                              q2_clean
0        how can i learn python  what is the best way to learn python
1  what is the capital of india    which city is the capital of india
2             how do i make tea                 how can i prepare tea
3      what is machine learning              explain machine learning
4  how can i improve my english   what are ways to improve my english


## TF-IDF Text Representation

In [14]:
all_questions = pd.concat([
    df["q1_clean"],
    df["q2_clean"]
])

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
vectorizer.fit(all_questions)

print("Total TF-IDF features:")
print(len(vectorizer.get_feature_names_out()))

Total TF-IDF features:
138


## Common Word Ratio

In [15]:
def common_word_ratio(q1, q2):
    words1 = set(q1.split())
    words2 = set(q2.split())

    common_words = len(words1 & words2)

    return common_words / max(len(words1), len(words2), 1)

## Cosine Similarity

In [16]:
def tfidf_similarity(q1, q2):
    vectors = vectorizer.transform([q1, q2])

    similarity = cosine_similarity(
        vectors[0], vectors[1]
    )[0][0]

    return similarity

## Feature Engineering

In [17]:
def create_features(q1, q2):
    return {
        "q1_len": len(q1),
        "q2_len": len(q2),
        "length_diff": abs(len(q1) - len(q2)),
        "common_word_ratio": common_word_ratio(q1, q2),
        "tfidf_similarity": tfidf_similarity(q1, q2)
    }


X = df.apply(
    lambda row: pd.Series(
        create_features(row["q1_clean"], row["q2_clean"])
    ),
    axis=1
)

y = df["is_duplicate"]

print("Extracted features:")
print(X.head())

Extracted features:
   q1_len  q2_len  length_diff  common_word_ratio  tfidf_similarity
0    22.0    36.0         14.0           0.250000          0.317424
1    28.0    34.0          6.0           0.714286          0.712465
2    17.0    21.0          4.0           0.600000          0.207490
3    24.0    24.0          0.0           0.500000          0.555663
4    28.0    35.0          7.0           0.428571          0.491135


## Train-Test Split

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 14
Testing samples: 6


## Train and Compare Multiple Models

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "SVM": SVC(kernel="linear")
}

best_model = None
best_accuracy = -1

for name, current_model in models.items():
    current_model.fit(X_train, y_train)

    predictions = current_model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(name, "Accuracy:", round(accuracy * 100, 2), "%")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = current_model

print("Best Model:", type(best_model).__name__)
print("Best Accuracy:", round(best_accuracy * 100, 2), "%")

Logistic Regression Accuracy: 66.67 %
Random Forest Accuracy: 66.67 %
SVM Accuracy: 66.67 %
Best Model: LogisticRegression
Best Accuracy: 66.67 %


## Evaluate the Best Model

In [24]:
predictions = best_model.predict(X_test)

print("Actual Values:")
print(y_test.tolist())

print("\nPredicted Values:")
print(predictions.tolist())

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Actual Values:
[0, 1, 0, 1, 1, 0]

Predicted Values:
[0, 0, 0, 1, 0, 0]

Confusion Matrix:
[[3 0]
 [2 1]]


## Predict a New Question Pair

In [25]:
q1 = "How do I learn Python?"
q2 = "What is the best way to learn Python?"

q1_clean = clean_text(q1)
q2_clean = clean_text(q2)

new_features = pd.DataFrame([
    create_features(q1_clean, q2_clean)
])

prediction = best_model.predict(new_features)[0]
similarity = tfidf_similarity(q1_clean, q2_clean)

print("Question 1:", q1)
print("Question 2:", q2)

print("Cosine Similarity:", round(similarity, 3))

if prediction == 1:
    print("Model Prediction: Similar / Duplicate Questions")
else:
    print("Model Prediction: Different Questions")


Question 1: How do I learn Python?
Question 2: What is the best way to learn Python?
Cosine Similarity: 0.341
Model Prediction: Similar / Duplicate Questions
